In [1]:
import math
import os
import sys
from datetime import datetime

import numpy as np
import pandas as pd
import torch

In [2]:
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from src.market.events import EventQueue, MarketEvent, OrderEvent, SignalEvent
from src.pipeline.data_handler import DataHandler
from src.strategy.model import Model
# from src.strategy.position.portfolio import Portfolio
from src.strategy.predict import Strategy
# from src.strategy.risk.risk_manager import RiskManager
from src.utility import get_config, get_path, read_file

In [4]:
config = get_config.load()
data = read_file.read_data("train", "data")

In [5]:
SYMBOLS = [symbol.split("/")[0] for symbol in config["pipeline"]["symbols"]]
date_format = "%Y-%m-%d %H:%M:%S"
START_DATE = datetime.strptime("2022-01-01 00:00:00", date_format)
END_DATE = datetime.strptime("2022-01-01 23:59:59", date_format)

capital = config["backtest"]["capital"]
transaction_cost_fraction = config["market"]["transaction_cost_fraction"]
bankruptcy_fraction = config["strategy"]["bankruptcy_fraction"]
slippage_cost_fraction = config["strategy"]["slippage_cost_fraction"]
stop_loss_multiple = config["strategy"]["stop_loss_multiple"]
stop_loss_portion = config["strategy"]["stop_loss_portion"]
take_profit_multiple = config["strategy"]["take_profit_multiple"]
take_profit_portion = config["strategy"]["take_profit_portion"]

seq_length = config["strategy"]["sequence_length"]
seq_length = 2
model_dir = get_path.absolute(config["path"]["strategy"]["model"])
portfolio_dir = get_path.absolute(config["path"]["strategy"]["portfolio"])
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [6]:
event_q = EventQueue()
data_handler = DataHandler(data, SYMBOLS, event_q, START_DATE, END_DATE)
model = Model(SYMBOLS, model_dir)
strategy = Strategy(data_handler, event_q, model, SYMBOLS, seq_length, device)

In [7]:
data_handler.get_latest_candle()

{}

In [8]:
data_handler.update_candles()

In [9]:
data_handler.get_latest_candle()

(open, BNB)                 511.500000
(high, BNB)                 517.800000
(low, BNB)                  511.400000
(close, BNB)                517.300000
(volume, BNB)             22537.945000
                              ...     
(vwap, SOL)                 172.644235
(mfi, SOL)                   33.376659
(log-return, SOL)             0.013962
(vol-spike, SOL)             -0.097222
(norm-volatility, SOL)        0.015315
Name: 2022-01-01 00:00:00, Length: 96, dtype: float64

In [10]:
for i in range(5):
    data_handler.update_candles()

In [11]:
data_handler.get_latest_candle()

(open, BNB)                 515.000000
(high, BNB)                 521.800000
(low, BNB)                  514.500000
(close, BNB)                518.000000
(volume, BNB)             23155.013000
                              ...     
(vwap, SOL)                 172.734874
(mfi, SOL)                   36.577367
(log-return, SOL)             0.010268
(vol-spike, SOL)             -0.290807
(norm-volatility, SOL)        0.018008
Name: 2022-01-01 05:00:00, Length: 96, dtype: float64

In [12]:
data_handler.get_latest_candle_datetime()

Timestamp('2022-01-01 05:00:00')

In [13]:
data_handler.get_latest_candles(5)

,"(open, BNB)","(high, BNB)","(low, BNB)","(close, BNB)","(volume, BNB)","(macd-signal-pct, BNB)","(macd-slope, BNB)","(sar, BNB)","(tema, BNB)","(ppo, BNB)",...,"(adx, SOL)","(stoch-rsi, SOL)","(bop, SOL)","(natr, SOL)","(obv, SOL)","(vwap, SOL)","(mfi, SOL)","(log-return, SOL)","(vol-spike, SOL)","(norm-volatility, SOL)"
timestamp,,,,,,,,,,,,,,,,,,,,,
2022-01-01 01:00:00,517.3,520.0,516.2,517.2,19858.246,0.000022,0.347162,-0.008382,0.007148,-0.356592,...,21.878117,13.973876,0.142292,1.281876,77539118.77,172.668662,40.802285,0.002028,0.197663,0.014647
2022-01-01 02:00:00,517.2,519.3,517.1,517.8,10209.673,0.000539,0.337537,-0.005413,0.005772,-0.418497,...,20.733553,1.621002,0.029851,1.217879,77556129.03,172.668043,36.595608,0.000116,-0.590311,0.003878
2022-01-01 03:00:00,517.8,518.8,517.3,518.1,9973.326,0.000898,0.302219,-0.003667,0.004210,-0.502750,...,19.635482,1.835789,0.281250,1.156078,77575114.25,172.677441,36.182999,0.001099,-0.540048,0.003701
2022-01-01 04:00:00,518.1,518.5,514.8,515.0,13355.034,0.000722,-0.000122,-0.008350,-0.001313,-0.573404,...,18.543217,-12.631871,-0.919255,1.149637,77553220.13,172.698660,31.410603,-0.008420,-0.462501,0.009388
2022-01-01 05:00:00,515.0,521.8,514.5,518.0,23155.013,0.000960,0.249613,0.023166,0.002834,-0.577940,...,17.729148,21.831078,0.567308,1.185241,77582080.62,172.734874,36.577367,0.010268,-0.290807,0.018008


In [14]:
data_handler.get_latest_candle_value(('open', 'BNB'))

np.float64(515.0)

In [15]:
data_handler.get_latest_candles_value(('close', 'BNB'), 5)

timestamp
2022-01-01 01:00:00    517.2
2022-01-01 02:00:00    517.8
2022-01-01 03:00:00    518.1
2022-01-01 04:00:00    515.0
2022-01-01 05:00:00    518.0
Name: (close, BNB), dtype: float64

In [16]:
event = MarketEvent(data_handler.get_latest_candle_datetime())

In [17]:
event

In [18]:
strategy.calculate_fiducia(event)

In [19]:
sevent = event_q.get_event()

In [20]:
sevent

In [21]:
sevent = event_q.get_event()
sevent

In [22]:
sevent = event_q.get_event()
sevent

In [23]:
sevent = event_q.get_event()
sevent

In [24]:
sevent = event_q.get_event()
sevent

In [25]:
sevent = event_q.get_event()
sevent

In [26]:
sevent = event_q.get_event()
sevent

In [27]:
sevent.fiducia

{'BNB': -0.20000000298023224,
 'BTC': 0.20000000298023224,
 'ETH': 0.20000000298023224,
 'SOL': -0.20000000298023224,
 'EXPOSURE': 1.0}

In [29]:
val = torch.sigmoid(torch.tensor(sevent.fiducia['EXPOSURE']))
val

tensor(0.7311)

In [30]:
sevent.fiducia['BNB'] * val

tensor(-0.1462)